# 09 final evaluation

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
from pathlib import Path
import sys, json, hashlib, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)

# Works when running from either the repository root or notebooks/.
def locate_repo_root():
    return PUBLICATION_ROOT

ROOT = locate_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features import MODEL_INPUTS

TEST_PATH = ROOT / 'data/notebook_outputs/ml_ready/test_ml.csv'
TRAIN_PATH = ROOT / 'data/notebook_outputs/ml_ready/train_ml.csv'
VALID_PATH = ROOT / 'data/notebook_outputs/ml_ready/validation_ml.csv'
RF_PATH = ROOT / 'models/random_forest_baseline.joblib'
TABPFN_PATH = ROOT / 'models/tabpfn_baseline.tabpfn_fit'
OUT = ROOT / 'results/final_evaluation'

# The saved Random Forest's input schema is the authoritative frozen schema.
# Do not assume a hard-coded feature count; verify it against the trained model.
from collections import Counter

project_features = list(MODEL_INPUTS)
rf_schema_check = joblib.load(RF_PATH)  # Read-only, no fitting or prediction.
trained_feature_names = getattr(rf_schema_check, 'feature_names_in_', None)
FEATURES = list(trained_feature_names) if trained_feature_names is not None else project_features

if not FEATURES:
    raise ValueError('No model input features were found.')
duplicates = [name for name, count in Counter(FEATURES).items() if count > 1]
if duplicates:
    raise ValueError(f'Duplicate feature names in frozen schema: {duplicates}')

expected_features = getattr(rf_schema_check, 'n_features_in_', None)
if expected_features is not None and len(FEATURES) != expected_features:
    raise ValueError(f'Saved RF model expects {expected_features} features, but schema contains {len(FEATURES)}.')

print('Current MODEL_INPUTS count:', len(project_features))
print('Saved Random Forest feature count:', expected_features)
print('Final evaluation features:', len(FEATURES))
if FEATURES != project_features:
    print('NOTE: Current MODEL_INPUTS differs from the saved RF feature schema; using the model\'s frozen feature order.')
if len(FEATURES) != 23:
    print('NOTE: Frozen schema does not contain 23 features. Verify this discrepancy against Notebook 07 before interpreting results.')

print('Project:', ROOT)
print('Model features:', len(FEATURES))
print('Held-out test file:', TEST_PATH)
print('Output directory:', OUT)

In [ ]:
# The test partition remains unopened in this cell.
for label, path in {
    'Held-out test CSV': TEST_PATH,
    'RF fitted model': RF_PATH,
    'TabPFN fitted model': TABPFN_PATH,
}.items():
    print(f'{label:23s}: {"FOUND" if path.is_file() else "MISSING"}  {path}')
    if not path.is_file():
        raise FileNotFoundError(path)

# Notebook 08 produced development-session Wazuh results here.
WAZUH_DEV_PATH = ROOT / 'data/notebook_outputs/wazuh_results/wazuh_combined_session_detections.csv'

# These are optional input locations, NOT files this notebook fabricates.
# A test Wazuh CSV must be generated using the unchanged Notebook 08 rules.
WAZUH_TEST_CANDIDATES = [
    ROOT / 'data/notebook_outputs/wazuh_results/wazuh_test_session_detections.csv',
    ROOT / 'results/wazuh/wazuh_test_session_detections.csv',
    ROOT / 'data/notebook_outputs/wazuh_results/wazuh_combined_session_detections.csv',
]

# This CSV must contain outputs of hybrid policies already fixed on development data.
# Supported names: Hybrid_OR, Hybrid_AND, Hybrid_ML_Guided (or a subset).
FROZEN_FUSION_TEST_CANDIDATES = [
    ROOT / 'data/notebook_outputs/hybrid/frozen_test_predictions.csv',
    ROOT / 'results/hybrid/frozen_test_predictions.csv',
]

print('\nExisting development Wazuh report:', WAZUH_DEV_PATH.is_file())
print('Possible test Wazuh files:')
for p in WAZUH_TEST_CANDIDATES:
    print(' ', 'FOUND' if p.is_file() else 'missing', p.relative_to(ROOT))
print('Possible frozen hybrid test-prediction files:')
for p in FROZEN_FUSION_TEST_CANDIDATES:
    print(' ', 'FOUND' if p.is_file() else 'missing', p.relative_to(ROOT))

# This notebook will not choose thresholds/delays by looking at test performance.

In [ ]:
# Run this and all later cells only once your validation-stage model/rule/fusion
# choices are frozen. Avoid changing those choices after seeing test performance.

test = pd.read_csv(TEST_PATH, low_memory=False)
required = {'SessionID', 'GroundTruthLabel', *FEATURES}
missing = sorted(required - set(test.columns))
assert not missing, f'Test CSV is missing required columns: {missing}'
assert len(test) == 286, f'Expected 286 held-out sessions; found {len(test)}.'
assert test['SessionID'].notna().all(), 'Some test SessionIDs are missing.'
assert not test['SessionID'].astype(str).duplicated().any(), 'Duplicate test SessionIDs.'

# Verify no overlap with saved train and validation sessions, if those files exist.
ids_test = set(test['SessionID'].astype(str))
for name, path in [('train', TRAIN_PATH), ('validation', VALID_PATH)]:
    if path.exists():
        other_ids = set(pd.read_csv(path, usecols=['SessionID'])['SessionID'].astype(str))
        assert not (ids_test & other_ids), f'{name}/test leakage: {len(ids_test & other_ids)} overlapping sessions.'

X_test = test.loc[:, FEATURES].apply(pd.to_numeric, errors='raise')
assert np.isfinite(X_test.to_numpy(dtype=float)).all(), 'Test model inputs contain missing or nonfinite values.'

y_test = pd.to_numeric(test['GroundTruthLabel'], errors='raise').astype(int).to_numpy()
assert set(np.unique(y_test)).issubset({0, 1}), 'GroundTruthLabel must be 0 or 1.'
assert set(np.unique(y_test)) == {0, 1}, 'ROC/PR curves require both classes in the test set.'

print('Test input shape:', X_test.shape)
print('Held-out session count:', len(test))
print('Ground-truth distribution (0 benign, 1 malicious):')
print(pd.Series(y_test).value_counts().sort_index().to_string())

In [ ]:
from tabpfn.model_loading import load_fitted_tabpfn_model

rf = joblib.load(RF_PATH)
tabpfn = load_fitted_tabpfn_model(TABPFN_PATH)

# Verify both fitted models against the frozen feature order before inference.
for model_name, model in [('RF', rf), ('TabPFN', tabpfn)]:
    if hasattr(model, 'n_features_in_'):
        assert model.n_features_in_ == X_test.shape[1], (
            f'{model_name} expects {model.n_features_in_} inputs; found {X_test.shape[1]}.'
        )
    if hasattr(model, 'feature_names_in_'):
        assert list(model.feature_names_in_) == FEATURES, f'{model_name} feature order mismatch.'

def model_predictions(model, X, name):
    classes = list(model.classes_)
    assert 0 in classes and 1 in classes, f'{name}: expected classes 0/1, found {classes}'
    labels = np.asarray(model.predict(X), dtype=int)
    probs = np.asarray(model.predict_proba(X))[:, classes.index(1)].astype(float)
    assert len(labels) == len(probs) == len(X)
    assert set(np.unique(labels)).issubset({0, 1})
    assert np.isfinite(probs).all() and ((probs >= 0) & (probs <= 1)).all()
    return labels, probs

rf_yhat, rf_score = model_predictions(rf, X_test, 'Random Forest')
tab_yhat, tab_score = model_predictions(tabpfn, X_test, 'TabPFN')

preds = pd.DataFrame({
    'SessionID': test['SessionID'].astype(str).to_numpy(),
    'ActualLabel': y_test,
    'RF_PredictedLabel': rf_yhat,
    'RF_MaliciousProbability': rf_score,
    'TabPFN_PredictedLabel': tab_yhat,
    'TabPFN_MaliciousProbability': tab_score,
})
print('Inference complete:', len(preds), 'held-out sessions')
display(preds.head())

In [ ]:
def read_complete_session_table(candidates, session_ids, name):
    problems = []
    for path in candidates:
        if not path.exists():
            continue
        df = pd.read_csv(path, low_memory=False)
        if 'SessionID' not in df:
            problems.append(f'{path.name}: no SessionID')
            continue
        df = df.copy()
        df['SessionID'] = df['SessionID'].astype(str)
        df = df[df['SessionID'].isin(session_ids)]
        if df['SessionID'].duplicated().any():
            problems.append(f'{path.name}: duplicated held-out SessionID')
            continue
        missing_ids = session_ids - set(df['SessionID'])
        if missing_ids:
            problems.append(f'{path.name}: missing {len(missing_ids)} of {len(session_ids)} test sessions')
            continue
        if 'Partition' in df.columns:
            if not df['Partition'].astype(str).str.lower().eq('test').all():
                problems.append(f'{path.name}: not explicitly labelled as test partition')
                continue
        print(f'{name}: {path.relative_to(ROOT)} ({len(df)} held-out sessions)')
        return path, df
    print(f'{name}: not available for all {len(session_ids)} held-out sessions')
    for problem in problems:
        print(' ', problem)
    return None, None

wazuh_path, wz = read_complete_session_table(
    WAZUH_TEST_CANDIDATES, ids_test, 'Wazuh rules'
)

# Only columns explicitly produced by the combined Wazuh detection pipeline
# are valid here. A single M01..M05 rule hit is NOT the same as the combined detector.
wazuh_columns = []
if wz is not None:
    wazuh_columns = [c for c in wz.columns if c.startswith('Wazuh_Detected_M05delay_')]
    if 'Wazuh_Detected' in wz:
        wazuh_columns = ['Wazuh_Detected'] + wazuh_columns
    if not wazuh_columns:
        print('No combined Wazuh detection flag exists in the saved test file.')
    else:
        wz = wz.set_index('SessionID').reindex(preds['SessionID'])
        for c in wazuh_columns:
            values = pd.to_numeric(wz[c], errors='raise')
            assert values.notna().all() and set(values.unique()).issubset({0, 1}), f'{c} is not a full binary flag.'
            preds[c] = values.to_numpy(dtype=int)
        print('Wazuh test policies available:', wazuh_columns)

In [ ]:
from pathlib import Path
import pandas as pd

test_ids = set(map(str, ids_test))

print(f"Required test sessions: {len(test_ids)}\n")

for candidate in WAZUH_TEST_CANDIDATES:
    path = Path(candidate)

    print(f"Checking: {path}")

    if not path.exists():
        print("  File does not exist\n")
        continue

    df = pd.read_csv(path)

    if "SessionID" not in df.columns:
        print("  SessionID column missing")
        print("  Available columns:", df.columns.tolist(), "\n")
        continue

    found_ids = set(df["SessionID"].dropna().astype(str))
    matched = test_ids.intersection(found_ids)

    print(f"  Matching test sessions: {len(matched)}/{len(test_ids)}")
    print("  Detection columns:", [
        col for col in df.columns
        if "wazuh" in col.lower() or "detect" in col.lower()
    ])
    print()


In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

assert len(preds) == 286, "Unexpected test session count"
assert preds["SessionID"].is_unique, "Duplicate test sessions"

y_true = preds["ActualLabel"].astype(int)

results = []

models = {
    "Random Forest": (
        "RF_PredictedLabel",
        "RF_MaliciousProbability"
    ),
    "TabPFN": (
        "TabPFN_PredictedLabel",
        "TabPFN_MaliciousProbability"
    )
}

for name, (pred_col, prob_col) in models.items():

    y_pred = preds[pred_col].astype(int)
    y_prob = preds[prob_col].astype(float)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Test Sessions": len(y_true),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1-Score": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "False Positive Rate": (
            fp / (fp + tn) if fp + tn else np.nan
        ),
        "ROC-AUC": (
            roc_auc_score(y_true, y_prob)
            if y_true.nunique() == 2 else np.nan
        ),
        "PR-AUC": (
            average_precision_score(y_true, y_prob)
            if y_true.nunique() == 2 else np.nan
        ),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })

ml_comparison = pd.DataFrame(results)

display(
    ml_comparison.style.format({
        col: "{:.4f}"
        for col in [
            "Accuracy", "Precision", "Recall",
            "F1-Score", "False Positive Rate",
            "ROC-AUC", "PR-AUC"
        ]
    })
)


In [ ]:
# Save final ML evaluation results

OUT.mkdir(parents=True, exist_ok=True)

ml_comparison.to_csv(
    OUT / "ml_test_comparison.csv",
    index=False
)

preds.to_csv(
    OUT / "ml_test_predictions.csv",
    index=False
)

# Identify the malicious session Random Forest missed
rf_false_negative = preds[
    (preds["ActualLabel"] == 1) &
    (preds["RF_PredictedLabel"] == 0)
]

print("Random Forest false negatives:", len(rf_false_negative))

display(
    rf_false_negative[
        [
            "SessionID",
            "ActualLabel",
            "RF_PredictedLabel",
            "RF_MaliciousProbability",
            "TabPFN_PredictedLabel",
            "TabPFN_MaliciousProbability"
        ]
    ]
)

print("Results saved to:", OUT)


In [ ]:
# --- CELL A: Reproduce Notebook 08 on its original development sessions ---
# Run after ML evaluation cells in your existing Notebook 09.
from pathlib import Path
import sys
import pandas as pd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wazuh_session_detection import (
    detect_wazuh_sessions, assert_dev_parity, COMBINED_COLUMN,
    M05_EXTRA_DELAY_SECONDS,
)

attributed_path = ROOT / 'data/notebook_outputs/wazuh_attributed_events_filtered.csv'
cutoffs_path = ROOT / 'data/notebook_outputs/session_prediction_cutoffs.csv'
archive_root = ROOT / 'data/wazuh/logs/archives'
saved_dev_path = ROOT / 'results/wazuh/wazuh_combined_session_detections.csv'

for required_path in (attributed_path, cutoffs_path, saved_dev_path):
    if not required_path.is_file():
        raise FileNotFoundError(f'Required Notebook 08 input/output missing: {required_path}')

attributed_events = pd.read_csv(attributed_path, low_memory=False)
cutoffs = pd.read_csv(cutoffs_path, low_memory=False)

train_sessions = pd.read_csv(TRAIN_PATH, usecols=['SessionID'])
validation_sessions = pd.read_csv(VALID_PATH, usecols=['SessionID'])
dev_sessions = pd.concat([
    train_sessions.assign(Partition='train'),
    validation_sessions.assign(Partition='validation'),
], ignore_index=True)
assert len(dev_sessions) == 890 and dev_sessions['SessionID'].is_unique, (
    'Development session population differs from Notebook 08'
)

# Labels are deliberately NOT passed to the detection algorithm.
dev_reproduced = detect_wazuh_sessions(
    dev_sessions[['SessionID', 'Partition']], attributed_events,
    cutoffs, archive_root=archive_root, lab_config=LAB_CONFIG,
)
original_dev = pd.read_csv(saved_dev_path, low_memory=False)
assert_dev_parity(dev_reproduced, original_dev)
print('PASS: Frozen M01-M05 decisions match Notebook 08 for all 890 development sessions.')



In [ ]:
# --- CELL B: Generate the 286 held-out Wazuh predictions (no retraining) ---
# Only run when Cell A passes completely. No test labels enter the detector.
test_sessions = pd.read_csv(TEST_PATH, usecols=['SessionID'])
assert len(test_sessions) == 286 and test_sessions['SessionID'].is_unique
assert set(test_sessions['SessionID'].astype(str)) == set(preds['SessionID'].astype(str))

test_sessions['Partition'] = 'test'
wazuh_test = detect_wazuh_sessions(
    test_sessions[['SessionID', 'Partition']], attributed_events,
    cutoffs, archive_root=archive_root, lab_config=LAB_CONFIG,
)
assert len(wazuh_test) == 286
assert set(wazuh_test['SessionID'].astype(str)) == set(preds['SessionID'].astype(str))

wazuh_path = ROOT / 'results/wazuh/wazuh_test_session_detections.csv'
wazuh_path.parent.mkdir(parents=True, exist_ok=True)
wazuh_test.to_csv(wazuh_path, index=False)

# Expose the exact same columns expected by the original Notebook 09 metrics cells.
wazuh_columns = [
    f'Wazuh_Detected_M05delay_{delay}s'
    for delay in M05_EXTRA_DELAY_SECONDS
]
wz = wazuh_test.set_index('SessionID').reindex(preds['SessionID'].astype(str))
assert not wz[wazuh_columns].isna().any().any()
for col in wazuh_columns:
    preds[col] = wz[col].to_numpy(dtype=int)

if 'hybrid_available' not in globals():
    hybrid_available = []  # No hybrid fusion policy is created or assumed.

print('PASS: Frozen Wazuh predictions generated for all 286 held-out sessions.')
print('Wazuh baseline:', COMBINED_COLUMN)
print('Detected:', int(preds[COMBINED_COLUMN].sum()))
print('Saved:', wazuh_path)
display(preds[['SessionID', 'ActualLabel', COMBINED_COLUMN]].head())



In [ ]:
# --- CELL C: Final RF vs TabPFN vs frozen Wazuh comparison ---
# Reuses the ml_comparison table you already calculated and saved.
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix,
)
import numpy as np

y_true = preds['ActualLabel'].astype(int)
y_wazuh = preds[COMBINED_COLUMN].astype(int)
tn, fp, fn, tp = confusion_matrix(y_true, y_wazuh, labels=[0, 1]).ravel()

wazuh_summary = pd.DataFrame([{
    'Model': 'Wazuh rules M01-M05 (0s M05 delay)',
    'Test Sessions': len(preds),
    'Accuracy': accuracy_score(y_true, y_wazuh),
    'Precision': precision_score(y_true, y_wazuh, zero_division=0),
    'Recall': recall_score(y_true, y_wazuh, zero_division=0),
    'F1-Score': f1_score(y_true, y_wazuh, zero_division=0),
    'False Positive Rate': fp / (fp + tn) if fp + tn else np.nan,
    'ROC-AUC': np.nan, 'PR-AUC': np.nan,
    'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
}])
final_comparison = pd.concat([ml_comparison, wazuh_summary], ignore_index=True)
display(final_comparison.round(4))

OUT.mkdir(parents=True, exist_ok=True)
final_comparison.to_csv(OUT / 'standalone_test_comparison.csv', index=False)
preds.to_csv(OUT / 'standalone_test_predictions.csv', index=False)
print('Final standalone results saved to:', OUT)
# Hybrid evaluation is intentionally not run without a pre-frozen fusion policy.


In [ ]:

# CELL D: Final Confusion Matrices and Error Analysis
# Random Forest, TabPFN and Wazuh Rule-Based Detection

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from pathlib import Path

# Output directory
OUT.mkdir(parents=True, exist_ok=True)

# Models and their prediction columns
models_to_evaluate = {
    "Random Forest": "RF_PredictedLabel",
    "TabPFN": "TabPFN_PredictedLabel",
    "Wazuh Rules": COMBINED_COLUMN
}

# Ground truth labels
y_true = preds["ActualLabel"].astype(int)

# Consistent appearance
labels = ["Benign", "Malicious"]
cmap = plt.get_cmap("Blues")

for model_name, prediction_column in models_to_evaluate.items():

    y_pred = preds[prediction_column].astype(int)

    # Calculate confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    tn, fp, fn, tp = cm.ravel()

    # Create figure
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    # Blue heatmap
    im = ax.imshow(
        cm,
        interpolation="nearest",
        cmap=cmap,
        vmin=0,
        vmax=max(int(cm.max()), 1)
    )

    # Axis labels
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(labels, fontsize=11)
    ax.set_yticklabels(labels, fontsize=11)

    ax.set_xlabel(
        "Predicted Label",
        fontsize=12,
        labelpad=12
    )

    ax.set_ylabel(
        "Actual Label",
        fontsize=12,
        labelpad=12
    )

    ax.set_title(
        f"{model_name}\nConfusion Matrix",
        fontsize=15,
        fontweight="bold",
        pad=18
    )

    # Add cell values with automatic text contrast
    for i in range(2):
        for j in range(2):

            value = cm[i, j]

            # Determine background brightness
            rgba = im.cmap(im.norm(value))
            luminance = (
                0.2126 * rgba[0] +
                0.7152 * rgba[1] +
                0.0722 * rgba[2]
            )

            text_color = (
                "white" if luminance < 0.5
                else "#1F2937"
            )

            ax.text(
                j, i,
                f"{value:,}",
                ha="center",
                va="center",
                fontsize=18,
                fontweight="bold",
                color=text_color
            )

    # Subtle grid lines between cells
    ax.set_xticks(np.arange(-0.5, 2, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 2, 1), minor=True)

    ax.grid(
        which="minor",
        color="white",
        linewidth=2
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    plt.tight_layout()

    # Save high-resolution image
    filename = model_name.lower().replace(" ", "_")

    fig.savefig(
        OUT / f"{filename}_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    # Identify incorrectly classified sessions
    error_mask = y_true.ne(y_pred)

    extra_cols = [
        c for c in [
            "RF_MaliciousProbability",
            "TabPFN_MaliciousProbability"
        ]
        if c in preds.columns
    ]

    errors = preds.loc[
        error_mask,
        [
            "SessionID",
            "ActualLabel",
            prediction_column,
            *extra_cols
        ]
    ].copy()

    errors["ErrorType"] = np.where(
        errors["ActualLabel"] == 0,
        "False Positive",
        "False Negative"
    )

    errors.to_csv(
        OUT / f"{filename}_errors.csv",
        index=False
    )

    # Save matrix values for reproducibility
    cm_df = pd.DataFrame(
        cm,
        index=["Actual Benign", "Actual Malicious"],
        columns=["Predicted Benign", "Predicted Malicious"]
    )

    cm_df.to_csv(
        OUT / f"{filename}_confusion_matrix.csv"
    )

    # Display results
    print(f"\n{'=' * 50}")
    print(f"{model_name}")
    print(f"{'=' * 50}")
    print(f"True Negatives:  {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Positives:  {tp}")
    print(f"Total Errors:    {len(errors)}")

    if not errors.empty:
        display(errors.head(10))
    else:
        print("No misclassified sessions.")

print("\nAll confusion matrices and error files saved to:")
print(OUT)


## Final Evaluation: Results and Discussion

### 1. Model Performance Comparison

The final evaluation was conducted using 286 held-out test sessions, consisting of 217 benign and 69 malicious sessions. Three detection approaches were evaluated: Random Forest, TabPFN and Wazuh rule-based detection.

TabPFN achieved the highest performance, correctly classifying all 286 sessions, with 100% accuracy, precision, recall and F1-score.

Random Forest achieved 99.65% accuracy and an F1-score of 99.27%. It correctly detected 68 of the 69 malicious sessions, producing one false negative and no false positives.

Wazuh rule-based detection achieved 90.91% accuracy, 100% recall and an F1-score of 84.15%. Although Wazuh detected all 69 malicious sessions, it incorrectly flagged 26 benign sessions, resulting in a false-positive rate of 11.98%.

### 2. Error Analysis

Random Forest missed one malicious session, assigning it a malicious probability of 0.28. TabPFN correctly classified the same session with a malicious probability of approximately 0.9988.

Wazuh detected all malicious sessions, including the one missed by Random Forest. However, its 26 false positives indicate that the existing rule-based approach produced more false alarms than either machine learning model.

These results suggest that the machine learning approaches were better at distinguishing benign and malicious behaviour within the simulated dataset, while Wazuh's rules achieved complete malicious-session coverage at the cost of lower precision.

### 3. System Integration

The prototype demonstrated that Windows Security and Sysmon events could be collected through Wazuh, processed into session-level behavioural features, classified by the trained Random Forest model and published to the Wazuh Dashboard.

Separate controlled experiments using university test accounts produced benign and malicious classifications that were successfully displayed in the dashboard.

This demonstrates operational integration between machine learning and Wazuh. However, it does not establish the performance of a separate hybrid decision-fusion algorithm.

### 4. Limitations

The evaluation used simulated university activity rather than real-world insider-threat incidents. The controlled nature of the dataset may have made behavioural patterns easier for the models to distinguish.

TabPFN's perfect test performance should therefore be interpreted cautiously. It demonstrates strong performance on the available held-out sessions but does not guarantee equivalent results in a real university environment.

The evaluation also used a single held-out test partition. Further evaluation on independently collected data would be necessary to establish generalisation.

During prototype testing, Wazuh event ingestion delays and incomplete telemetry were observed. These issues could affect detection availability in operational environments.

Finally, although the project demonstrated successful ML-Wazuh integration, no separately frozen hybrid fusion policy was evaluated on the held-out test set.

### 5. Conclusion

TabPFN achieved the strongest classification performance on the held-out test dataset, followed closely by Random Forest. Wazuh rule-based detection identified every malicious session but generated substantially more false positives.

The findings demonstrate the potential of machine learning to improve the precision of insider-threat detection in this simulated environment, while also highlighting the value of rule-based detection and the importance of reliable event collection.

Further testing using independent behavioural data is needed before drawing conclusions about real-world deployment performance.

In [ ]:

# CELL E: Hybrid Detection and Alert Prioritisation
# Combines existing Random Forest and Wazuh decisions
# Exploratory analysis; no retraining or threshold tuning

import numpy as np
import pandas as pd

required = [
    "SessionID",
    "ActualLabel",
    "RF_PredictedLabel",
    "RF_MaliciousProbability",
    COMBINED_COLUMN
]

missing = [c for c in required if c not in preds.columns]
assert not missing, f"Missing columns: {missing}"
assert preds["SessionID"].is_unique, "Duplicate session IDs"

fusion = preds[required].copy()

for col in ["ActualLabel", "RF_PredictedLabel", COMBINED_COLUMN]:
    fusion[col] = pd.to_numeric(
        fusion[col], errors="raise"
    ).astype(int)

    assert fusion[col].isin([0, 1]).all(), (
        f"{col} contains non-binary values"
    )

# Hybrid decision function
# 1 = suspicious/malicious; 0 = benign

def hybrid_priority(wazuh_flag, rf_flag):
    if wazuh_flag == 1 and rf_flag == 1:
        return "High Priority"
    elif wazuh_flag != rf_flag:
        return "Needs Review"
    else:
        return "Low Risk"

# Apply fusion to each test session
# Ground-truth labels are NOT used to assign priorities

fusion["HybridPriority"] = [
    hybrid_priority(w, r)
    for w, r in zip(
        fusion[COMBINED_COLUMN],
        fusion["RF_PredictedLabel"]
    )
]

fusion["HighPriorityAlert"] = (
    fusion["HybridPriority"] == "High Priority"
).astype(int)

fusion["ReviewRequired"] = (
    fusion["HybridPriority"] != "Low Risk"
).astype(int)

fusion["AnalysisType"] = "exploratory_post_hoc"

# Evaluate distribution against ground truth
order = ["High Priority", "Needs Review", "Low Risk"]

hybrid_summary = (
    fusion.groupby("HybridPriority")["ActualLabel"]
    .agg(Total="size", Malicious="sum")
    .reindex(order, fill_value=0)
)

hybrid_summary["Benign"] = (
    hybrid_summary["Total"] - hybrid_summary["Malicious"]
)

hybrid_summary = hybrid_summary[
    ["Total", "Malicious", "Benign"]
]

print("HYBRID ALERT PRIORITISATION RESULTS")
display(hybrid_summary)

print(
    "\nTotal sessions:",
    len(fusion)
)

print(
    "High-priority alerts:",
    int(hybrid_summary.loc["High Priority", "Total"])
)

print(
    "Sessions requiring review:",
    int(fusion["ReviewRequired"].sum())
)

# Save results
OUT.mkdir(parents=True, exist_ok=True)

fusion.to_csv(
    OUT / "hybrid_priority_exploratory_sessions.csv",
    index=False
)

hybrid_summary.to_csv(
    OUT / "hybrid_priority_exploratory_summary.csv"
)

print("\nResults saved to:", OUT)


In [ ]:

# CELL F: Hybrid vs Wazuh Alert Workload Comparison
# Exploratory evaluation of alert prioritisation

import pandas as pd

y_true = fusion["ActualLabel"].astype(int)

approaches = {
    "Wazuh Rules": fusion[COMBINED_COLUMN].astype(int),
    "Hybrid - High Priority Only": fusion["HighPriorityAlert"].astype(int),
    "Hybrid - All Review Alerts": fusion["ReviewRequired"].astype(int)
}

results = []

for name, flags in approaches.items():
    tp = int(((y_true == 1) & (flags == 1)).sum())
    fp = int(((y_true == 0) & (flags == 1)).sum())
    fn = int(((y_true == 1) & (flags == 0)).sum())

    results.append({
        "Approach": name,
        "Alerts": tp + fp,
        "TruePositives": tp,
        "FalsePositives": fp,
        "FalseNegatives": fn,
        "Precision": tp / (tp + fp) if tp + fp else 0,
        "Recall": tp / (tp + fn) if tp + fn else 0
    })

workload_comparison = pd.DataFrame(results)

display(workload_comparison.round(4))

wazuh_alerts = int(fusion[COMBINED_COLUMN].sum())
high_priority = int(fusion["HighPriorityAlert"].sum())
total_reviews = int(fusion["ReviewRequired"].sum())

print("\nHYBRID WORKLOAD ANALYSIS")
print(f"Wazuh alerts: {wazuh_alerts}")
print(f"High-priority alerts: {high_priority}")
print(f"Additional reviews: {total_reviews - high_priority}")
print(f"Total hybrid reviews: {total_reviews}")

high_reduction = (
    100 * (wazuh_alerts - high_priority) / wazuh_alerts
    if wazuh_alerts else 0
)

total_reduction = (
    100 * (wazuh_alerts - total_reviews) / wazuh_alerts
    if wazuh_alerts else 0
)

print(f"\nReduction in high-priority workload: {high_reduction:.2f}%")
print(f"Reduction in total review workload: {total_reduction:.2f}%")

workload_comparison.to_csv(
    OUT / "hybrid_workload_comparison_exploratory.csv",
    index=False
)

print("\nComparison saved to:", OUT)
